In [1]:
%%bash
cat << 'EOF' > super_pi.c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char **argv) {
    long long accuracy = 20; 
    if (argc > 1) accuracy = atoll(argv[1]);
    
    // Reduced the base multiplier slightly so the accuracy numbers are easier to tune
    long long iterations = 2000000LL * accuracy; 
    
    double a = 1.0, pi = 0.0;
    for (long long i = 0; i < iterations; i++) {
        pi += a / (2 * i + 1);
        a = -a;
    }

    // Force the compiler to keep the loop by outputting the result to stderr
    fprintf(stderr, "Calculation complete. (PI ~ %.5f)\n", pi * 4);

    // Read exact scheduler metrics from the kernel before exiting
    FILE *f = fopen("/proc/self/schedstat", "r");
    if (f) {
        unsigned long long exec_ns, wait_ns, slices;
        if (fscanf(f, "%llu %llu %llu", &exec_ns, &wait_ns, &slices) == 3) {
            printf("%llu,%llu,%llu\n", exec_ns, wait_ns, slices);
        }
        fclose(f);
    }
    return 0;
}
EOF

gcc -O3 super_pi.c -o super_pi
chmod +x super_pi
echo "Fixed Super PI benchmark compiled successfully."

Fixed Super PI benchmark compiled successfully.


In [1]:
import subprocess
import time
import numpy as np
import pandas as pd
from IPython.display import display
# -----------------------
def run_workload(total_runs, num_procs, accuracy, csv_filename):
    print(f"Executing {total_runs} benchmark runs.")
    print(f"Workload: {num_procs} processes, accuracy {accuracy}")
    print("-" * 105)
    
    all_runs_data = []
    
    for run in range(1, total_runs + 1):
        processes = []
        total_start = time.time()
        
        # taskset -c 0 pins all processes to CPU 0 to force scheduler contention
        # cmd = ["taskset", "-c", "0", "./super_pi", accuracy]

        cmd = ["./super_pi", accuracy]
    
        # Spawn processes and capture their stdout
        for _ in range(num_procs):
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, text=True)
            processes.append(p)
    
        turnaround_times = []
        wait_latencies = []
        context_switches = []
    
        # Wait for completion and parse the kernel metrics
        for p in processes:
            out, _ = p.communicate()
            turnaround_times.append(time.time() - total_start)
            
            if out.strip():
                try:
                    # Parse the CSV output from our C program
                    exec_ns, wait_ns, slices = map(int, out.strip().split(','))
                    wait_latencies.append(wait_ns / 1e9)  # Convert nanoseconds to seconds
                    context_switches.append(slices)
                except ValueError:
                    pass
    
        # --- Macro-Metrics ---
        std_dev = np.std(turnaround_times)                 # Fairness
        avg_turnaround = np.mean(turnaround_times)         # Responsiveness
        max_turnaround = np.max(turnaround_times)          # System completion time
        throughput = num_procs / max_turnaround            # Tasks per second
        
        # --- Micro-Metrics ---
        avg_latency = np.mean(wait_latencies) if wait_latencies else 0       # Avg time spent starving
        avg_switches = np.mean(context_switches) if context_switches else 0  # Avg times scheduled
        total_switches = np.sum(context_switches) if context_switches else 0 # Total context switches
    
        run_label = f"Run {run:02d}" + (" (Warm-up)" if run == 1 else "")
        
        all_runs_data.append({
            "Run": run_label,
            "Std_Dev_s": round(std_dev, 4),
            "Avg_Turn_s": round(avg_turnaround, 4),
            "Throughput_tps": round(throughput, 4),
            "Avg_Latency_s": round(avg_latency, 4),
            "Avg_Context_Switches": round(avg_switches, 1),
            "Total_Context_Switches": total_switches
        })
    
        print(f"{run_label:<18} | StdDev: {std_dev:.4f}s | Latency: {avg_latency:.4f}s | Switches: {total_switches} | Throughput: {throughput:.2f} tps")
    
    print("-" * 105)
    print(f"Testing complete. Saving data to {csv_filename}...")
    
    # Export to CSV and display a clean table in Jupyter
    df = pd.DataFrame(all_runs_data)
    df.to_csv(csv_filename, index=False)
    display(df)

In [3]:
# --- Test Parameters ---
num_procs = 100     # Number of concurrent processes
accuracy = "2000"    # Workload intensity
total_runs = 5    # 1 warm-up + 4 recorded runs

# IMPORTANT: Change this filename when you test the eBPF scheduler!
csv_filename = "scx_simple_metrics.csv" 
run_workload(total_runs, num_procs, accuracy, csv_filename)

Executing 5 benchmark runs.
Workload: 100 processes, accuracy 2000
---------------------------------------------------------------------------------------------------------


Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
C

Run 01 (Warm-up)   | StdDev: 0.0411s | Latency: 19.1463s | Switches: 61626 | Throughput: 3.10 tps


Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
C

Run 02             | StdDev: 0.0604s | Latency: 19.1225s | Switches: 61619 | Throughput: 3.10 tps


...

Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
C

Run 04             | StdDev: 0.1152s | Latency: 18.9850s | Switches: 61488 | Throughput: 3.10 tps


Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
Calculation complete. (PI ~ 3.14159)
C

Run 05             | StdDev: 0.0838s | Latency: 19.0597s | Switches: 61576 | Throughput: 3.10 tps
---------------------------------------------------------------------------------------------------------
Testing complete. Saving data to scx_simple_metrics.csv...


,Run,Std_Dev_s,Avg_Turn_s,Throughput_tps,Avg_Latency_s,Avg_Context_Switches,Total_Context_Switches
0,Run 01 (Warm-up),0.0411,32.1713,3.0982,19.1463,616.3,61626
1,Run 02,0.0604,32.1351,3.1023,19.1225,616.2,61619
2,Run 03,0.1144,32.0860,3.0876,18.9515,614.4,61441
3,Run 04,0.1152,32.0784,3.0972,18.9850,614.9,61488
4,Run 05,0.0838,32.1039,3.0997,19.0597,615.8,61576
